# 03 — 多臂老虎机 (Multi-Armed Bandit)

**本教程在课程中的位置**

| 前置知识 | 本讲内容 | 后续内容 |
|----------|----------|----------|
| Python + NumPy 基础 | 多臂老虎机问题 | MDP + Bellman 方程（04） |
| 概率论基础 | 探索-利用权衡 | 动态规划（05） |
| 02 数学准备 | 经典算法实现 | Monte Carlo（06） |

**天数**：第 2 天（与 02 数学准备共享一天）
**预计时间**：3 小时（理论 1.5h + 实验 1.5h）


## 学习目标

完成本 Notebook 后，你将能够：

1. **定义** K-臂老虎机问题的形式化表示：动作集合、奖励分布、遗憾
2. **推导** 增量式动作价值更新公式
3. **实现** 五种经典探索策略并从零编写算法：
   - 贪心 (Greedy)
   - ε-贪心 (ε-greedy)
   - 乐观初始化 (Optimistic Initialization)
   - 上置信界 (UCB)
   - 梯度老虎机 (Gradient Bandit)
4. **定量分析** 探索率 ε 对收敛速度和最终性能的影响
5. **可视化** 探索-利用权衡曲线并解释不同算法的行为差异
6. **批判性思考** 每种算法的优缺点、适用场景和调参方向


In [ ]:
import sys
import os

# 确保项目根目录在 Python 路径中
PROJECT_ROOT = "/workspace/data/vggt-omega/rl"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

%matplotlib inline

# 输出目录
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs/figures")
os.makedirs(OUT_DIR, exist_ok=True)

# 可视化配置
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 100,
    "savefig.bbox": "tight",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.figsize": (10, 5),
})

print("环境就绪")


## 1. K-臂老虎机问题

### 1.1 问题形式化

**多臂老虎机** (Multi-Armed Bandit) 是强化学习中最简单的设定——只有一个状态，智能体只需在每个时间步选择一个动作。

**数学定义**：

- $K$ 个动作（臂），动作集 $\mathcal{A} = \{a_1, a_2, \dots, a_K\}$
- 每个动作 $a$ 对应一个未知的奖励分布 $p(r \mid a)$，其期望值为 $\mathbb{E}[R \mid a] = q_*(a)$
- 在时间步 $t$，智能体选择动作 $A_t$，环境返回随机奖励 $R_t \sim p(r \mid A_t)$
- 智能体的目标：在 $T$ 步内**最大化累积奖励** $\sum_{t=1}^T R_t$

### 1.2 遗憾 (Regret)

由于各动作的真实期望 $q_*(a)$ 未知，智能体需要通过尝试来学习。引入**遗憾** (Regret) 来衡量算法性能：

$$
\mu^* = \max_a q_*(a) \quad \text{(最优动作的期望奖励)}
$$
$$
\text{Regret}_t = \mu^* \cdot t - \sum_{i=1}^t R_i
$$

遗憾值衡量的是 "如果从一开始就知道最优动作，我们能多拿多少奖励"。

### 1.3 探索-利用权衡 (Exploration-Exploitation Dilemma)

- **利用 (Exploitation)**：选择当前已知的最佳动作（最大化即时奖励）
- **探索 (Exploration)**：选择非最优动作（收集信息，可能发现更好的动作）

核心矛盾：探索牺牲短期收益以获取长期增益。平衡这一矛盾是所有 bandit 算法的核心问题。


## 2. 动作价值估计

### 2.1 真实价值与估计价值

动作 $a$ 的**真实价值** $q_*(a)$ 是未知的。智能体维护一个**估计价值** $Q_t(a)$，表示到时间步 $t$ 为止对 $q_*(a)$ 的最佳猜测。

### 2.2 样本平均法 (Sample-Average Method)

最直观的估计方式：取该动作所有已观察奖励的**平均值**。

$$
Q_t(a) = \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbf{1}\{A_i = a\}}{\sum_{i=1}^{t-1} \mathbf{1}\{A_i = a\}}
$$

其中 $\mathbf{1}\{\cdot\}$ 是指示函数。

当分母趋于无穷时，由大数定律，$Q_t(a) \to q_*(a)$。

### 2.3 增量式更新

存储所有历史奖励再求和效率太低。我们可以推导**增量式更新公式**：

设 $Q_n$ 为某动作被选择 $n$ 次后的估计价值。

$$
\begin{aligned}
Q_{n+1} &= \frac{1}{n} \sum_{i=1}^n R_i \\
&= \frac{1}{n} \left( R_n + \sum_{i=1}^{n-1} R_i \right) \\
&= \frac{1}{n} \left( R_n + (n-1) Q_n \right) \\
&= Q_n + \frac{1}{n} \left( R_n - Q_n \right)
\end{aligned}
$$

这个形式简洁优美：

$$
\boxed{Q_{n+1} = Q_n + \frac{1}{n} [R_n - Q_n]}
$$

- $[R_n - Q_n]$ 是**误差项** (error)
- $\frac{1}{n}$ 是**学习率** (step size)

这是强化学习中**最核心的更新模式**——「预测 $\to$ 观察误差 $\to$ 修正预测」——后续所有 TD 方法都是这一形式的推广。


## 3. 代码实现：样本平均 Bandit Agent

下面我们实现一个通用的 Bandit Agent 基类，以及具体的贪心策略。


In [ ]:
from rl_course.envs.bandit import MultiArmedBandit


class BanditAgent:
    """通用 Bandit 智能体基类。

    维护每个动作的价值估计 Q 和选择计数 N。
    子类需实现 select_action() 方法。

    Args:
        k: 动作（臂）数量
        initial_q: 初始价值估计
    """

    def __init__(self, k: int, initial_q: float = 0.0):
        self.k = k
        self.initial_q = initial_q
        self.Q = np.full(k, initial_q, dtype=np.float64)
        self.N = np.zeros(k, dtype=np.int32)
        self.step = 0

    def select_action(self) -> int:
        """选择动作（由子类实现）"""
        raise NotImplementedError

    def update(self, action: int, reward: float) -> None:
        """增量更新动作价值估计 Q(action)。

        使用公式: Q_{n+1} = Q_n + (1/n)(R_n - Q_n)
        """
        self.step += 1
        self.N[action] += 1
        n = self.N[action]
        self.Q[action] += (reward - self.Q[action]) / n

    def reset(self) -> None:
        """重置智能体状态"""
        self.Q = np.full(self.k, self.initial_q, dtype=np.float64)
        self.N = np.zeros(self.k, dtype=np.int32)
        self.step = 0

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(k={self.k})"


print("BanditAgent 基类定义完成")


In [ ]:
class GreedyAgent(BanditAgent):
    """贪心策略：总是选择当前估计价值最大的动作。"""

    def select_action(self) -> int:
        # 如有多个最大值，随机选一个（打破平局）
        return int(np.random.choice(np.where(self.Q == self.Q.max())[0]))


# 快速测试
np.random.seed(42)
env = MultiArmedBandit(k=5, seed=42)
agent = GreedyAgent(k=5)
print("真实价值:", env.true_means)
print("最优动作:", env.optimal_action)

for t in range(100):
    a = agent.select_action()
    r = env.pull(a)
    agent.update(a, r)

print("估计价值:", agent.Q)
print("选择次数:", agent.N)
print("累积遗憾:", env.regret)


## 4. 探索策略

前面贪心策略的问题很明显：**永远不会探索**。一旦某个动作初始表现好，贪心就会一直选择它，可能错失真正的最优动作。

以下是几种经典的探索策略：

| 策略 | 核心思想 | 关键参数 | 特点 |
|------|----------|----------|------|
| **贪心 (Greedy)** | 选当前最优 | — | 容易陷入局部最优 |
| **ε-贪心 (ε-greedy)** | 以概率 ε 随机探索 | ε | 简单有效，最常用 |
| **乐观初始化 (Optimistic Init)** | 初始 Q 设高，鼓励探索 | $Q_0$ | 早期自动探索，后期退化为贪心 |
| **UCB** | 按不确定性加成的上限选动作 | $c$ | 理论保证，确定性探索 |
| **梯度老虎机 (Gradient Bandit)** | 学习动作偏好，softmax 选动作 | $\alpha$ | 基于偏好而非价值估计 |


### 4.1 ε-贪心 (Epsilon-Greedy)

以概率 $1-\varepsilon$ 选择贪心动作，以概率 $\varepsilon$ 均匀随机选择一个动作：

$$
A_t =
\begin{cases}
\arg\max_a Q_t(a) & \text{概率 } 1-\varepsilon \\[4pt]
\text{均匀随机动作} & \text{概率 } \varepsilon
\end{cases}
$$


In [ ]:
class EpsilonGreedyAgent(BanditAgent):
    """ε-贪心策略。

    Args:
        k: 臂数
        epsilon: 探索概率 (0 <= epsilon <= 1)
        initial_q: 初始价值估计
    """

    def __init__(self, k: int, epsilon: float = 0.1, initial_q: float = 0.0):
        super().__init__(k, initial_q)
        self.epsilon = epsilon

    def select_action(self) -> int:
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k)   # 探索：均匀随机
        else:
            return int(np.random.choice(np.where(self.Q == self.Q.max())[0]))  # 利用


print("EpsilonGreedyAgent 定义完成")


### 4.2 实验：比较不同 ε 值

我们设计一个实验，固定环境种子，比较 $\varepsilon = 0,\ 0.01,\ 0.1$ 的性能。


In [ ]:
def run_experiment(env, agent, n_steps=1000):
    """单次实验：在给定环境上运行一个 agent。

    Returns:
        dict 包含 rewards, regrets, optimal_frac（每步记录）
    """
    env.reset()
    agent.reset()
    rewards = np.zeros(n_steps)
    regrets = np.zeros(n_steps)
    optimal_frac = np.zeros(n_steps)
    opt_action = env.optimal_action

    for t in range(n_steps):
        a = agent.select_action()
        r = env.pull(a)
        agent.update(a, r)
        rewards[t] = r
        regrets[t] = env.regret
        n_opt = agent.N[opt_action]
        optimal_frac[t] = n_opt / (t + 1)

    return {"rewards": rewards, "regrets": regrets, "optimal_frac": optimal_frac}


def run_multiple_runs(env, agent_class, agent_kwargs,
                      n_steps=1000, n_runs=50, env_seed=42):
    """多次运行取平均，消除随机噪声。"""
    all_rewards = np.zeros((n_runs, n_steps))
    all_regrets = np.zeros((n_runs, n_steps))
    all_optimal = np.zeros((n_runs, n_steps))

    for run in range(n_runs):
        run_env = MultiArmedBandit(
            k=env.k, reward_type=env.reward_type,
            true_means=env.true_means.copy(),
            sigma=env.sigma, seed=env_seed + run,
        )
        agent = agent_class(**agent_kwargs)
        result = run_experiment(run_env, agent, n_steps)
        all_rewards[run] = result["rewards"]
        all_regrets[run] = result["regrets"]
        all_optimal[run] = result["optimal_frac"]

    return {
        "avg_reward": all_rewards.mean(axis=0),
        "avg_regret": all_regrets.mean(axis=0),
        "avg_optimal": all_optimal.mean(axis=0),
    }


print("实验辅助函数定义完成")


In [ ]:
# 创建环境（10 臂，Bernoulli 奖励）
np.random.seed(42)
base_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)

epsilons = [0.0, 0.01, 0.1]
n_steps = 1000
n_runs = 50

results = {}
for eps in epsilons:
    print(f"Running epsilon={eps} ...")
    results[eps] = run_multiple_runs(
        env=base_env,
        agent_class=EpsilonGreedyAgent,
        agent_kwargs={"k": 10, "epsilon": eps, "initial_q": 0.0},
        n_steps=n_steps, n_runs=n_runs,
    )

print("epsilon 对比实验完成")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["red", "green", "blue"]

for idx, eps in enumerate(epsilons):
    res = results[eps]
    steps = np.arange(n_steps)
    axes[0].plot(steps, res["avg_reward"], label=f"epsilon={eps}", color=colors[idx])
    axes[1].plot(steps, res["avg_regret"], label=f"epsilon={eps}", color=colors[idx])
    axes[2].plot(steps, res["avg_optimal"], label=f"epsilon={eps}", color=colors[idx])

axes[0].set_title("平均奖励 (Average Reward)")
axes[0].set_xlabel("时间步"); axes[0].set_ylabel("平均奖励")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("累积遗憾 (Cumulative Regret)")
axes[1].set_xlabel("时间步"); axes[1].set_ylabel("遗憾")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].set_title("最优动作比例 (% Optimal Action)")
axes[2].set_xlabel("时间步"); axes[2].set_ylabel("比例")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

fig.suptitle("epsilon-贪心策略对比 (K=10, Bernoulli, 50 次运行平均)", fontsize=15)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_epsilon_comparison.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


### 4.3 结果分析

从上面的曲线可以观察到：

| $\varepsilon$ | 最终最优动作比例 | 累积遗憾 | 特点 |
|:---:|:---:|:---:|------|
| **0.0** (纯贪心) | ~30-40% | 最高 | 初始快速上升但很快停滞，被次优臂困住 |
| **0.01** | ~80-90% | 最低 | 少量探索使它能找到最优臂，且不损失太多 |
| **0.1** | ~90-95% | 中等 | 探索充分但浪费了约 10% 的步骤在随机选择上 |

**关键洞见**：
- 零探索（$\varepsilon=0$）看似"最努力"，但实际表现最差——因为它很可能被困在次优解
- 少量探索（$\varepsilon=0.01$）在长期表现最好
- 过多探索（$\varepsilon=0.1$）虽然找到了最优，但浪费太多步骤在随机探索上


### 4.4 乐观初始化 (Optimistic Initialization)

**核心思想**：把初始价值估计设得非常高，让智能体一开始"失望"，从而自动去探索其他动作。

例如，Bernoulli 奖励范围是 $[0, 1]$，我们把初始值设为 $Q_1(a) = 5$。第一次选任何臂都会得到远低于预期的奖励，智能体会认为"这个臂不如预期，换一个试试"。

**优点**：简单，不需要额外参数（除了初始值本身）
**缺点**：只在早期探索，后期退化为贪心；对非平稳环境不适应


In [ ]:
class OptimisticAgent(BanditAgent):
    """乐观初始化智能体：贪心 + 高初始值。"""

    def __init__(self, k: int, initial_q: float = 5.0):
        super().__init__(k, initial_q)

    def select_action(self) -> int:
        return int(np.random.choice(np.where(self.Q == self.Q.max())[0]))


# --- 实验 ---
np.random.seed(42)
base_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
n_steps = 1000
n_runs = 50

opt_results = run_multiple_runs(
    env=base_env, agent_class=OptimisticAgent,
    agent_kwargs={"k": 10, "initial_q": 5.0},
    n_steps=n_steps, n_runs=n_runs,
)

eps01_res = run_multiple_runs(
    env=base_env, agent_class=EpsilonGreedyAgent,
    agent_kwargs={"k": 10, "epsilon": 0.1, "initial_q": 0.0},
    n_steps=n_steps, n_runs=n_runs,
)

eps001_res = run_multiple_runs(
    env=base_env, agent_class=EpsilonGreedyAgent,
    agent_kwargs={"k": 10, "epsilon": 0.01, "initial_q": 0.0},
    n_steps=n_steps, n_runs=n_runs,
)

print("乐观初始化实验完成")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
steps = np.arange(n_steps)

axes[0].plot(steps, opt_results["avg_reward"], label="乐观初始化 (Q0=5)", color="orange")
axes[0].plot(steps, eps01_res["avg_reward"], label="epsilon-贪心 (eps=0.1)", color="blue", alpha=0.7)
axes[0].plot(steps, eps001_res["avg_reward"], label="epsilon-贪心 (eps=0.01)", color="green", alpha=0.7)
axes[0].set_title("平均奖励"); axes[0].set_xlabel("时间步")
axes[0].set_ylabel("平均奖励"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, opt_results["avg_optimal"], label="乐观初始化 (Q0=5)", color="orange")
axes[1].plot(steps, eps01_res["avg_optimal"], label="epsilon-贪心 (eps=0.1)", color="blue", alpha=0.7)
axes[1].plot(steps, eps001_res["avg_optimal"], label="epsilon-贪心 (eps=0.01)", color="green", alpha=0.7)
axes[1].set_title("最优动作比例"); axes[1].set_xlabel("时间步")
axes[1].set_ylabel("比例"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle("乐观初始化 vs epsilon-贪心 (K=10, Bernoulli, 50 次运行平均)", fontsize=14)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_optimistic_vs_epsilon.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


### 4.5 乐观初始化分析

乐观初始化的关键特性：

1. **早期自动探索**：高初始值驱使智能体访问所有臂至少一次
2. **探索量不可控**：探索完全由初始值决定，不能像 ε-贪心那样持续探索
3. **适合平稳环境**：在真实价值不变的场景下效果好
4. **对初始值敏感**：$Q_0$ 太小就失去效果，太大会导致前几次选择完全随机

> 注意：乐观初始化在非平稳环境（臂的奖励分布随时间变化）中效果很差，因为一旦估计值收敛，就没有机制继续探索了。


### 4.6 UCB (Upper Confidence Bound)

UCB 采用一种更智能的策略：**根据不确定性选择动作**。

如果一个动作被选择的次数很少，我们对它的估计就很不确定，就应该给它"机会"。

UCB1 算法的动作选择规则：

$$
A_t = \arg\max_a \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right]
$$

其中：
- $Q_t(a)$：当前价值估计（利用项）
- $c \sqrt{\frac{\ln t}{N_t(a)}}$：**置信上界**（探索项）
  - $c$：探索系数，控制探索强度
  - $t$：总步数
  - $N_t(a)$：动作 $a$ 被选的次数
  - $\sqrt{\ln t / N_t(a)}$：当 $N_t(a)$ 小时，这一项大，鼓励探索

**霍夫丁不等式**保证：$Q_t(a) + c\sqrt{\ln t / N_t(a)}$ 是 $q_*(a)$ 的高置信度上界。

UCB 的优势是**确定性探索**——不需要随机抽样，而是通过置信边界自动平衡探索与利用。


In [ ]:
class UCBAgent(BanditAgent):
    """UCB1 智能体。

    Args:
        k: 臂数
        c: 探索系数（越大探索越强）
        initial_q: 初始价值估计
    """

    def __init__(self, k: int, c: float = 2.0, initial_q: float = 0.0):
        super().__init__(k, initial_q)
        self.c = c

    def select_action(self) -> int:
        # 确保每个臂至少被选一次
        unselected = np.where(self.N == 0)[0]
        if len(unselected) > 0:
            return int(unselected[0])

        t = self.step + 1
        ucb_values = self.Q + self.c * np.sqrt(np.log(t) / self.N)
        return int(np.random.choice(np.where(ucb_values == ucb_values.max())[0]))


# 快速验证
np.random.seed(42)
env_test = MultiArmedBandit(k=5, seed=42)
ucb_agent = UCBAgent(k=5, c=2.0)
for t in range(200):
    a = ucb_agent.select_action()
    r = env_test.pull(a)
    ucb_agent.update(a, r)

print("真实价值:", env_test.true_means)
print("估计价值:", ucb_agent.Q)
print("选择次数:", ucb_agent.N)
print("累积遗憾:", env_test.regret)


In [ ]:
# --- UCB 实验 ---
np.random.seed(42)
base_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
n_steps = 1000
n_runs = 50

c_values = [0.5, 2.0, 5.0]
ucb_results = {}
for c in c_values:
    print(f"Running UCB with c={c} ...")
    ucb_results[c] = run_multiple_runs(
        env=base_env, agent_class=UCBAgent,
        agent_kwargs={"k": 10, "c": c},
        n_steps=n_steps, n_runs=n_runs,
    )

print("UCB 实验完成")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
steps = np.arange(n_steps)
ucb_colors = ["purple", "red", "brown"]

for idx, c in enumerate(c_values):
    res = ucb_results[c]
    axes[0].plot(steps, res["avg_reward"], label=f"UCB (c={c})", color=ucb_colors[idx])
    axes[1].plot(steps, res["avg_optimal"], label=f"UCB (c={c})", color=ucb_colors[idx])

# epsilon=0.01 作为参考
axes[0].plot(steps, eps001_res["avg_reward"], label="epsilon-贪心 (eps=0.01)", color="green", alpha=0.7, ls="--")
axes[1].plot(steps, eps001_res["avg_optimal"], label="epsilon-贪心 (eps=0.01)", color="green", alpha=0.7, ls="--")

axes[0].set_title("平均奖励"); axes[0].set_xlabel("时间步")
axes[0].set_ylabel("平均奖励"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("最优动作比例"); axes[1].set_xlabel("时间步")
axes[1].set_ylabel("比例"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle("UCB 策略对比 (K=10, Bernoulli, 50 次运行平均)", fontsize=14)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_ucb_comparison.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


## 5. 增量式更新：深入理解

在前面我们推导了增量式更新公式：

$$
Q_{n+1} = Q_n + \frac{1}{n}(R_n - Q_n)
$$

这是一个**通用的更新模式**，后续在整个课程中会反复出现：

$$
\text{新估计} = \text{旧估计} + \text{步长} \times (\text{目标} - \text{旧估计})
$$

在更一般的形式中，步长 $\frac{1}{n}$ 可以替换为 $\alpha$（常数学习率），形成指数衰减平均：

$$
Q_{n+1} = Q_n + \alpha (R_n - Q_n)
$$

两种步长方案的对比：

| 步长方案 | 公式 | 特性 |
|----------|------|------|
| 样本平均 | $\alpha_n = 1/n$ | 每个样本等权，最终收敛到均值 |
| 常数步长 | $\alpha_n = \alpha$ | 指数衰减旧样本权重，适合非平稳环境 |


## 6. 探索-利用权衡可视化

下面我们通过直方图直观展示不同算法在"探索"和"利用"上的行为差异。


In [ ]:
np.random.seed(42)
vis_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
n_steps = 500

agents = {
    "贪心 (eps=0)": EpsilonGreedyAgent(k=10, epsilon=0.0),
    "eps-贪心 (eps=0.1)": EpsilonGreedyAgent(k=10, epsilon=0.1),
    "乐观初始化 (Q0=5)": OptimisticAgent(k=10, initial_q=5.0),
    "UCB (c=2)": UCBAgent(k=10, c=2.0),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, agent) in enumerate(agents.items()):
    env_copy = MultiArmedBandit(
        k=10, reward_type="bernoulli",
        true_means=vis_env.true_means.copy(), seed=99,
    )
    agent.reset()

    action_history = np.zeros(n_steps, dtype=np.int32)
    for t in range(n_steps):
        a = agent.select_action()
        r = env_copy.pull(a)
        agent.update(a, r)
        action_history[t] = a

    ax = axes[idx]
    action_counts = np.bincount(action_history, minlength=10)
    bar_colors = ["#e74c3c" if i == env_copy.optimal_action else "#3498db" for i in range(10)]
    ax.bar(range(10), action_counts, color=bar_colors, edgecolor="white")
    ax.axvline(x=env_copy.optimal_action, color="red", ls="--", lw=2,
               label=f"最优臂 (mu={env_copy.true_means[env_copy.optimal_action]:.2f})")
    ax.set_title(f"{name}  最优率: {action_counts[env_copy.optimal_action]/n_steps:.1%}")
    ax.set_xlabel("动作"); ax.set_ylabel("选择次数")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

fig.suptitle("探索-利用权衡：500 步后各臂选择分布 (红色 = 最优臂)", fontsize=14)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_exploration_exploitation_tradeoff.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


## 7. 梯度老虎机 (Gradient Bandit)

### 7.1 核心思想

前面几种方法都基于**动作价值估计**。梯度老虎机采用了不同的思路：**学习动作偏好 (preference)**。

每个动作 $a$ 有一个偏好值 $H_t(a)$，动作被选择的概率由 **softmax** 分布给出：

$$
\pi_t(a) = \Pr(A_t = a) = \frac{e^{H_t(a)}}{\sum_{b=1}^K e^{H_t(b)}}
$$

### 7.2 随机梯度上升更新

偏好值 $H_t(a)$ 通过**随机梯度上升**更新，目标是最大化期望奖励：

$$
\begin{aligned}
H_{t+1}(A_t) &= H_t(A_t) + \alpha (R_t - \bar{R}_t)(1 - \pi_t(A_t)) \\
H_{t+1}(a)   &= H_t(a)   - \alpha (R_t - \bar{R}_t) \pi_t(a), \quad \forall a \neq A_t
\end{aligned}
$$

其中：
- $\alpha > 0$：学习率
- $\bar{R}_t$：到时间 $t$ 为止的平均奖励（作为**基线**，减少方差）
- $(R_t - \bar{R}_t)$：**相对奖励**——高于基线时增强所选动作的偏好，低于基线时减弱

这种更新本质上是**策略梯度方法**在 Bandit 设定下的特例——后续在 Policy Gradient 章节会再次见到这一形式。


In [ ]:
class GradientBanditAgent:
    """梯度老虎机智能体。

    使用 softmax 偏好和随机梯度上升更新。

    Args:
        k: 臂数
        alpha: 学习率
        use_baseline: 是否使用平均奖励作为基线
        initial_h: 初始偏好值
    """

    def __init__(self, k: int, alpha: float = 0.1,
                 use_baseline: bool = True, initial_h: float = 0.0):
        self.k = k
        self.alpha = alpha
        self.use_baseline = use_baseline
        self.H = np.full(k, initial_h, dtype=np.float64)
        self.pi = np.full(k, 1.0 / k, dtype=np.float64)
        self.avg_reward = 0.0
        self.step = 0

    def select_action(self) -> int:
        H_shifted = self.H - self.H.max()
        exp_H = np.exp(H_shifted)
        self.pi = exp_H / exp_H.sum()
        return int(np.random.choice(self.k, p=self.pi))

    def update(self, action: int, reward: float) -> None:
        self.step += 1
        if self.use_baseline:
            self.avg_reward += (reward - self.avg_reward) / self.step
            baseline = self.avg_reward
        else:
            baseline = 0.0
        relative = reward - baseline
        one_hot = np.zeros(self.k)
        one_hot[action] = 1.0
        self.H += self.alpha * relative * (one_hot - self.pi)

    def reset(self) -> None:
        self.H = np.full(self.k, 0.0, dtype=np.float64)
        self.pi = np.full(self.k, 1.0 / self.k, dtype=np.float64)
        self.avg_reward = 0.0
        self.step = 0


# 快速验证
np.random.seed(42)
env_gb = MultiArmedBandit(k=5, seed=42)
gb_agent = GradientBanditAgent(k=5, alpha=0.1)
for t in range(200):
    a = gb_agent.select_action()
    r = env_gb.pull(a)
    gb_agent.update(a, r)

print("真实价值:", env_gb.true_means)
print("最终偏好:", gb_agent.H)
print("选择概率:", gb_agent.pi)
print("最优动作:", env_gb.optimal_action)


In [ ]:
def run_gradient_experiment(env, agent, n_steps=1000):
    """梯度 Bandit 单次实验"""
    env.reset()
    agent.reset()
    rewards = np.zeros(n_steps)
    regrets = np.zeros(n_steps)
    optimal_frac = np.zeros(n_steps)
    opt_action = env.optimal_action

    for t in range(n_steps):
        a = agent.select_action()
        r = env.pull(a)
        agent.update(a, r)
        rewards[t] = r
        regrets[t] = env.regret
        n_opt = env.action_counts[opt_action]
        optimal_frac[t] = n_opt / (t + 1)
    return {"rewards": rewards, "regrets": regrets, "optimal_frac": optimal_frac}


def run_multiple_gradient(env, agent_kwargs, n_steps=1000, n_runs=50, env_seed=42):
    """多次运行梯度 Bandit 取平均"""
    all_rewards = np.zeros((n_runs, n_steps))
    all_regrets = np.zeros((n_runs, n_steps))
    all_optimal = np.zeros((n_runs, n_steps))

    for run in range(n_runs):
        run_env = MultiArmedBandit(
            k=env.k, reward_type=env.reward_type,
            true_means=env.true_means.copy(),
            sigma=env.sigma, seed=env_seed + run,
        )
        agent = GradientBanditAgent(**agent_kwargs)
        result = run_gradient_experiment(run_env, agent, n_steps)
        all_rewards[run] = result["rewards"]
        all_regrets[run] = result["regrets"]
        all_optimal[run] = result["optimal_frac"]

    return {
        "avg_reward": all_rewards.mean(axis=0),
        "avg_regret": all_regrets.mean(axis=0),
        "avg_optimal": all_optimal.mean(axis=0),
    }


# --- 梯度 Bandit 实验 ---
np.random.seed(42)
base_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
n_steps = 1000
n_runs = 50

alphas = [0.01, 0.1, 0.5]
grad_results = {}
for alpha in alphas:
    print(f"Running Gradient Bandit with alpha={alpha} ...")
    grad_results[alpha] = run_multiple_gradient(
        env=base_env,
        agent_kwargs={"k": 10, "alpha": alpha, "use_baseline": True},
        n_steps=n_steps, n_runs=n_runs,
    )

print("梯度 Bandit 实验完成")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
steps = np.arange(n_steps)
grad_colors = ["teal", "coral", "slateblue"]

for idx, alpha in enumerate(alphas):
    res = grad_results[alpha]
    axes[0].plot(steps, res["avg_reward"], label=f"Gradient (alpha={alpha})", color=grad_colors[idx])
    axes[1].plot(steps, res["avg_optimal"], label=f"Gradient (alpha={alpha})", color=grad_colors[idx])

axes[0].set_title("平均奖励"); axes[0].set_xlabel("时间步")
axes[0].set_ylabel("平均奖励"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("最优动作比例"); axes[1].set_xlabel("时间步")
axes[1].set_ylabel("比例"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle("梯度老虎机对比 (K=10, Bernoulli, 50 次运行平均)", fontsize=14)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_gradient_bandit_comparison.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


## 8. 所有方法大对比

下面把所有算法放在一起比较——包括学习曲线和参数敏感性分析。


In [ ]:
# --- 统一对比 ---
np.random.seed(42)
base_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
n_steps = 1000
n_runs = 50

all_configs = [
    ("贪心 (eps=0)", EpsilonGreedyAgent, {"k": 10, "epsilon": 0.0}),
    ("eps-贪心 (eps=0.01)", EpsilonGreedyAgent, {"k": 10, "epsilon": 0.01}),
    ("eps-贪心 (eps=0.1)", EpsilonGreedyAgent, {"k": 10, "epsilon": 0.1}),
    ("乐观初始化 (Q0=5)", OptimisticAgent, {"k": 10, "initial_q": 5.0}),
    ("UCB (c=2)", UCBAgent, {"k": 10, "c": 2.0}),
    ("Gradient (alpha=0.1)", GradientBanditAgent, {"k": 10, "alpha": 0.1, "use_baseline": True}),
]

all_results = {}
for name, cls, kwargs in all_configs:
    print(f"Running {name} ...")
    if cls is GradientBanditAgent:
        all_results[name] = run_multiple_gradient(
            env=base_env, agent_kwargs=kwargs,
            n_steps=n_steps, n_runs=n_runs,
        )
    else:
        all_results[name] = run_multiple_runs(
            env=base_env, agent_class=cls, agent_kwargs=kwargs,
            n_steps=n_steps, n_runs=n_runs,
        )

print("所有方法实验完成")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
steps = np.arange(n_steps)
palette = plt.cm.Set2(np.linspace(0, 1, len(all_configs)))

for idx, (name, cls, kwargs) in enumerate(all_configs):
    res = all_results[name]
    color = palette[idx]
    axes[0, 0].plot(steps, res["avg_reward"], label=name, color=color, linewidth=1.5)
    axes[0, 1].plot(steps, res["avg_regret"], label=name, color=color, linewidth=1.5)
    axes[1, 0].plot(steps, res["avg_optimal"], label=name, color=color, linewidth=1.5)
    final_reward = res["avg_reward"][-100:].mean()
    axes[1, 1].bar(name, final_reward, color=color, alpha=0.8)

axes[0, 0].set_title("平均奖励"); axes[0, 0].set_xlabel("时间步")
axes[0, 0].set_ylabel("平均奖励"); axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True, alpha=0.3)
axes[0, 1].set_title("累积遗憾"); axes[0, 1].set_xlabel("时间步")
axes[0, 1].set_ylabel("遗憾"); axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].set_title("最优动作比例"); axes[1, 0].set_xlabel("时间步")
axes[1, 0].set_ylabel("比例"); axes[1, 0].legend(fontsize=8); axes[1, 0].grid(True, alpha=0.3)
axes[1, 1].set_title("最终性能 (最后 100 步平均奖励)")
axes[1, 1].set_ylabel("平均奖励"); axes[1, 1].tick_params(axis="x", rotation=30); axes[1, 1].grid(True, alpha=0.3)

fig.suptitle("所有 Bandit 方法对比 (K=10, Bernoulli, 50 次运行平均)", fontsize=15)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_all_methods_comparison.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


### 8.1 参数敏感性分析

每种方法都有关键超参数，这里我们探究参数变化对性能的影响。


In [ ]:
# --- 参数扫描实验 ---
np.random.seed(42)
base_env = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
n_steps = 500
n_runs = 30

def parameter_sweep(agent_class, param_name, param_values, fixed_kwargs,
                    env, n_steps, n_runs):
    """对某个参数进行扫描实验"""
    results = {}
    for val in param_values:
        kwargs = {**fixed_kwargs, param_name: val}
        if agent_class is GradientBanditAgent:
            res = run_multiple_gradient(env, kwargs, n_steps, n_runs)
        else:
            res = run_multiple_runs(env, agent_class, kwargs, n_steps, n_runs)
        results[val] = {
            "final_optimal": res["avg_optimal"][-100:].mean(),
            "final_reward": res["avg_reward"][-100:].mean(),
        }
    return results

# epsilon 扫描
eps_sweep = parameter_sweep(
    EpsilonGreedyAgent, "epsilon",
    [0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5],
    {"k": 10}, base_env, n_steps, n_runs,
)

# UCB c 扫描
ucb_sweep = parameter_sweep(
    UCBAgent, "c",
    [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    {"k": 10}, base_env, n_steps, n_runs,
)

# Gradient alpha 扫描
grad_sweep = parameter_sweep(
    GradientBanditAgent, "alpha",
    [0.01, 0.05, 0.1, 0.2, 0.5, 1.0],
    {"k": 10, "use_baseline": True}, base_env, n_steps, n_runs,
)

# 乐观初始化 Q0 扫描
opt_sweep = parameter_sweep(
    OptimisticAgent, "initial_q",
    [0.0, 0.5, 1.0, 2.0, 5.0, 10.0],
    {"k": 10}, base_env, n_steps, n_runs,
)

print("参数扫描完成")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. epsilon 敏感度
eps_keys = list(eps_sweep.keys())
eps_vals = [eps_sweep[k]["final_optimal"] for k in eps_keys]
axes[0, 0].plot(range(len(eps_keys)), eps_vals, "o-", color="blue", linewidth=2)
axes[0, 0].set_xticks(range(len(eps_keys)))
axes[0, 0].set_xticklabels([str(v) for v in eps_keys])
axes[0, 0].set_title("epsilon-贪心: epsilon 对最优动作比例的影响")
axes[0, 0].set_xlabel("epsilon"); axes[0, 0].set_ylabel("最终最优动作比例")
axes[0, 0].grid(True, alpha=0.3); axes[0, 0].axhline(y=1.0, color="gray", ls="--", alpha=0.5)

# 2. UCB c 敏感度
ucb_keys = list(ucb_sweep.keys())
ucb_vals = [ucb_sweep[k]["final_optimal"] for k in ucb_keys]
axes[0, 1].plot(range(len(ucb_keys)), ucb_vals, "o-", color="purple", linewidth=2)
axes[0, 1].set_xticks(range(len(ucb_keys)))
axes[0, 1].set_xticklabels([str(v) for v in ucb_keys])
axes[0, 1].set_title("UCB: c 对最优动作比例的影响")
axes[0, 1].set_xlabel("c"); axes[0, 1].set_ylabel("最终最优动作比例")
axes[0, 1].grid(True, alpha=0.3); axes[0, 1].axhline(y=1.0, color="gray", ls="--", alpha=0.5)

# 3. Gradient alpha 敏感度
grad_keys = list(grad_sweep.keys())
grad_vals = [grad_sweep[k]["final_optimal"] for k in grad_keys]
axes[1, 0].plot(range(len(grad_keys)), grad_vals, "o-", color="teal", linewidth=2)
axes[1, 0].set_xticks(range(len(grad_keys)))
axes[1, 0].set_xticklabels([str(v) for v in grad_keys])
axes[1, 0].set_title("Gradient Bandit: alpha 对最优动作比例的影响")
axes[1, 0].set_xlabel("alpha"); axes[1, 0].set_ylabel("最终最优动作比例")
axes[1, 0].grid(True, alpha=0.3); axes[1, 0].axhline(y=1.0, color="gray", ls="--", alpha=0.5)

# 4. Optimistic Q0 敏感度
opt_keys = list(opt_sweep.keys())
opt_vals = [opt_sweep[k]["final_optimal"] for k in opt_keys]
axes[1, 1].plot(range(len(opt_keys)), opt_vals, "o-", color="orange", linewidth=2)
axes[1, 1].set_xticks(range(len(opt_keys)))
axes[1, 1].set_xticklabels([str(v) for v in opt_keys])
axes[1, 1].set_title("乐观初始化: Q0 对最优动作比例的影响")
axes[1, 1].set_xlabel("初始 Q0"); axes[1, 1].set_ylabel("最终最优动作比例")
axes[1, 1].grid(True, alpha=0.3); axes[1, 1].axhline(y=1.0, color="gray", ls="--", alpha=0.5)

fig.suptitle("参数敏感性分析 (K=10, Bernoulli, 30 次运行平均)", fontsize=14)
fig.tight_layout()
fpath = os.path.join(OUT_DIR, "03_parameter_sensitivity.png")
fig.savefig(fpath)
plt.show()
print("已保存:", fpath)


## 9. 总结

### 9.1 算法对比总结

| 方法 | 探索方式 | 关键参数 | 收敛速度 | 最终性能 | 适用场景 |
|------|----------|----------|:--------:|:--------:|----------|
| **贪心 (Greedy)** | 无 | — | 快 | 差 | 不推荐独立使用 |
| **ε-贪心 (ε-greedy)** | 随机 | $\varepsilon$ | 中 | 好 | 通用选择，简单可靠 |
| **乐观初始化 (Optimistic)** | 初始值驱动 | $Q_0$ | 快 | 中 | 平稳环境，短期实验 |
| **UCB** | 确定性 | $c$ | 中 | 很好 | 需要理论保证的场景 |
| **梯度老虎机 (Gradient)** | 概率型 | $\alpha$ | 稍慢 | 好 | 价值估计不准时 |

### 9.2 核心公式回顾

| 概念 | 公式 |
|------|------|
| 样本平均估计 | $Q_n(a) = \frac{1}{n}\sum_{i=1}^n R_i$ |
| 增量式更新 | $Q_{n+1} = Q_n + \alpha_n(R_n - Q_n)$ |
| ε-贪心选择 | $A_t = \begin{cases} \arg\max Q & \text{概率 } 1-\varepsilon \\ \text{随机} & \text{概率 } \varepsilon \end{cases}$ |
| UCB 选择 | $A_t = \arg\max_a \left[ Q_t(a) + c\sqrt{\frac{\ln t}{N_t(a)}} \right]$ |
| Softmax 策略 | $\pi_t(a) = e^{H_t(a)} / \sum_b e^{H_t(b)}$ |
| 梯度更新 | $H_{t+1}(a) = H_t(a) + \alpha(R_t - \bar{R}_t)(\mathbf{1}\{a=A_t\} - \pi_t(a))$ |

### 9.3 关键直觉

1. **探索是必要的**：零探索的贪心策略在复杂问题中几乎必然失败
2. **更多探索 ≠ 更好**：过度探索浪费时间，关键是找到适量探索
3. **探索方式很重要**：确定性探索（UCB）往往比随机探索（ε-贪心）更高效
4. **增量式更新是 RL 的基石**：几乎所有 RL 算法都遵循"预测 → 误差 → 修正"的框架
5. **基线减少方差**：梯度 Bandit 中的平均奖励基线与后续 Policy Gradient 中的 Advantage 函数一脉相承


## 10. 面试准备问题

以下问题覆盖了 Bandit 问题的核心概念，是 RL 面试中的常见题：

### 基础概念
1. **什么是探索-利用权衡？为什么它是 RL 的核心挑战？**
2. **解释 regret 的定义及其作为评估指标的优缺点。**
3. **为什么贪心策略可能在简单环境中也表现不佳？**

### 算法原理
4. **推导增量式更新公式 $Q_{n+1} = Q_n + \frac{1}{n}(R_n - Q_n)$。**
5. **比较 ε-贪心 和 UCB 的探索方式的本质区别。**
6. **乐观初始化为什么能促进探索？它的局限性是什么？**
7. **解释 UCB 公式中 $\sqrt{\ln t / N_t(a)}$ 项的含义。为什么要有 $\ln t$？**

### 梯度方法
8. **梯度 Bandit 与基于价值的 Bandit 方法有什么本质不同？**
9. **为什么梯度 Bandit 中要用基线（平均奖励）？**
10. **梯度 Bandit 的更新律与策略梯度定理有何关系？**

### 深入思考
11. **在非平稳环境中（臂的奖励分布随时间变化），上述方法需要如何调整？**
12. **如果有 10000 个臂但只有 1000 次尝试，应该用什么策略？**
13. **如何扩展 Bandit 算法到上下文 Bandit (Contextual Bandit)？**


## 11. 练习题

### 练习 1：推导验证 (简单)
使用纸上推导验证增量式更新公式。假设一个动作被选了 5 次，奖励分别为 $[1, 0, 1, 1, 0]$。
1. 用样本平均法计算 $Q_1, Q_2, Q_3, Q_4, Q_5$
2. 用增量式更新公式重复计算
3. 两种结果一致吗？

### 练习 2：修改 ε-贪心 (中等)
修改 `EpsilonGreedyAgent`，实现随时间衰减的 ε 值：
$$
\varepsilon_t = \max(\varepsilon_{\min}, \varepsilon_{\text{start}} \times \text{decay}^t)
$$
比较衰减版本与固定 ε 版本在 2000 步内的表现。

### 练习 3：实现非平稳环境实验 (中等)
修改 `MultiArmedBandit`，让每个臂的真实均值每 200 步随机游走一次：
```python
self.true_means += np.random.randn(self.k) * 0.01
```
比较 ε-贪心（固定 ε）vs 常数步长（$\alpha=0.1$）的增量更新在非平稳环境下的表现。

### 练习 4：多臂赌博机仿真 (较难)
设计一个实验：
- K=50 臂，Gaussian 奖励 $N(\mu_k, 1)$，$\mu_k \sim N(0, 1)$
- 在 2000 步内比较 UCB, ε-贪心, Gradient Bandit 的最终 regret
- 画出不同参数配置下的 Pareto 前沿（探索量 vs 最终奖励）

### 练习 5：推导证明 (较难)
证明 UCB 选择公式 $A_t = \arg\max_a \left[ Q_t(a) + \sqrt{\frac{2\ln t}{N_t(a)}} \right]$ 的 regret 上界为 $O(\log t)$。

提示：使用 Hoeffding 不等式和 Union Bound。


## 12. 拓展阅读

### 经典教材
- **Sutton & Barto, "Reinforcement Learning: An Introduction" (2nd ed.)**
  - Chapter 2: Multi-armed Bandits — Bandit 问题的权威参考
- **Lattimore & Szepesvari, "Bandit Algorithms"**
  - 更深入的理论分析，含 regret 上界证明

### 经典论文
- **Auer, Cesa-Bianchi & Fischer (2002)**: "Finite-time Analysis of the Multiarmed Bandit Problem"
  - UCB 算法的原始论文
- **Kuleshov & Precup (2014)**: "Algorithms for Multi-Armed Bandit Problems"
  - 实验性比较，适合入门

### 进阶话题
- **Contextual Bandit**：结合上下文的 Bandit（如 LinUCB），在推荐系统中广泛应用
- **Thompson Sampling**：贝叶斯方法，在实践中通常优于 UCB
- **Adversarial Bandit**：对抗性设定下的 Bandit（如 EXP3 算法）
- **Combinatorial Bandit**：每次选择一组动作的组合 Bandit

### 在线资源
- [Bandit Algorithms GitHub](https://github.com/johnmyleswhite/BanditsBook) — 配套代码
- [RL Course @ David Silver](https://www.davidsilver.uk/teaching/) — Lecture 1 对应 Bandit
- [OpenAI Spinning Up](https://spinningup.openai.com/) — RL 入门教程

---

> **下一讲预告**：04_MDP.ipynb — 我们将从 Bandit 的"单步决策"扩展到"序列决策"，引入马尔可夫决策过程 (MDP) 和 Bellman 方程。


## 附录：快速参考表

| 符号 | 含义 |
|------|------|
| $K$ | 臂的数量 |
| $A_t$ | 时间 $t$ 选择的动作 |
| $R_t$ | 时间 $t$ 获得的奖励 |
| $q_*(a)$ | 动作 $a$ 的真实价值 |
| $Q_t(a)$ | 时间 $t$ 对动作 $a$ 的估计价值 |
| $N_t(a)$ | 到时间 $t$ 为止选择动作 $a$ 的次数 |
| $\varepsilon$ | ε-贪心的探索概率 |
| $c$ | UCB 的探索系数 |
| $\alpha$ | 学习率 / 步长 |
| $H_t(a)$ | 动作 $a$ 在时间 $t$ 的偏好值 |
| $\pi_t(a)$ | 在时间 $t$ 选择动作 $a$ 的概率 |
| $\bar{R}_t$ | 到时间 $t$ 为止的平均奖励（基线） |
